# Model Tuning — Grid Search Comparison
เปรียบเทียบ model หลายตัวด้วย Grid Search หาตัวที่ F1 ดีที่สุด

## 1. Imports

In [ ]:
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.sparse import hstack, csr_matrix

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import SGDClassifier, LogisticRegression
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    classification_report, f1_score,
    roc_auc_score, precision_score, recall_score
)

from toxic_pipeline import (
    ALL_ENGINEERED_FEATURE_COLUMNS,
    WORD_TFIDF_CONFIG as PIPELINE_WORD_TFIDF_CONFIG,
    build_all_engineered_features,
    clean_text,
    protect_non_toxic_negations,
)

print('All imports OK')


## 1.5 Constants


In [ ]:
DATA_PATH = '../data/train.csv'
LABEL_COLUMNS_TO_DROP = ['severe_toxic', 'obscene', 'threat', 'insult', 'identity_hate']

SEARCH_RANDOM_STATE = 42
SEARCH_TEST_SIZE = 0.20
SEARCH_CV_SPLITS = 3
SEARCH_CLASS_WEIGHTS = ['balanced', {0:1, 1:3}]
search_cv = StratifiedKFold(
    n_splits=SEARCH_CV_SPLITS,
    shuffle=True,
    random_state=SEARCH_RANDOM_STATE,
)

WORD_TFIDF_CONFIG = dict(PIPELINE_WORD_TFIDF_CONFIG)
SCALER_CONFIG = {'with_mean': False}
SGD_FIXED_MAX_ITER = 500
LR_FIXED_MAX_ITER = 500

SGD_GRID_PARAM_GRID = [
    {
        'alpha': [1e-5, 1e-4, 1e-3],
        'penalty': ['l2', 'l1'],
        'class_weight': SEARCH_CLASS_WEIGHTS,
        'max_iter': [SGD_FIXED_MAX_ITER],
        'tol': [1e-4],
    },
    {
        'alpha': [1e-5, 1e-4],
        'penalty': ['elasticnet'],
        'class_weight': SEARCH_CLASS_WEIGHTS,
        'max_iter': [SGD_FIXED_MAX_ITER],
        'tol': [1e-4],
        'l1_ratio': [0.15, 0.50],
    },
]
SGD_GRID_MODEL_CONFIG = {
    'loss': 'modified_huber',
    'random_state': SEARCH_RANDOM_STATE,
    'n_jobs': None,
}
LR_GRID_PARAM_GRID = [
    {
        'C': [0.03, 0.30, 3.0],
        'class_weight': SEARCH_CLASS_WEIGHTS,
        'solver': ['liblinear'],
        'penalty': ['l2', 'l1'],
        'tol': [1e-4],
    },
    {
        'C': [0.10, 1.0],
        'class_weight': SEARCH_CLASS_WEIGHTS,
        'solver': ['saga'],
        'penalty': ['l2', 'l1'],
        'tol': [1e-4],
    },
]
LR_GRID_MODEL_CONFIG = {
    'max_iter': LR_FIXED_MAX_ITER,
    'random_state': SEARCH_RANDOM_STATE,
    'n_jobs': None,
}

GRID_SEARCH_WORD_VECTORIZER_PATH = 'grid_search_word_vectorizer.pkl'
GRID_SEARCH_SCALER_PATH = 'grid_search_scaler.pkl'
GRID_SEARCH_BEST_MODEL_PATH = 'grid_search_best_model.pkl'


## 2. Load & Clean Data

In [ ]:
df = pd.read_csv(DATA_PATH)
df = df.drop(columns=LABEL_COLUMNS_TO_DROP)

df['raw_text'] = df['comment_text'].fillna('').astype(str)
df['protected_text'] = df['raw_text'].apply(protect_non_toxic_negations)
df['clean_text'] = df['protected_text'].apply(clean_text)
df = df[df['clean_text'] != ''].copy()
df = df.drop_duplicates(subset=['clean_text']).reset_index(drop=True)

print(f'Dataset shape: {df.shape}')
print(df['toxic'].value_counts())


## 3. Feature Engineering

In [ ]:
engineered_df = pd.concat(
    [build_all_engineered_features(text) for text in df['raw_text']],
    ignore_index=True,
)
df[ALL_ENGINEERED_FEATURE_COLUMNS] = engineered_df

print('Feature engineering done')


## 4. Vectorize + Scale + Split

In [ ]:
ENG_FEATURE_COLS = ALL_ENGINEERED_FEATURE_COLUMNS.copy()

X_clean = df['clean_text']
X_eng = df[ENG_FEATURE_COLS]
y = df['toxic']

(
    X_clean_train, X_clean_test,
    X_eng_train, X_eng_test,
    y_train, y_test
) = train_test_split(
    X_clean, X_eng, y,
    test_size=SEARCH_TEST_SIZE,
    random_state=SEARCH_RANDOM_STATE,
    stratify=y,
)

word_vec = TfidfVectorizer(**WORD_TFIDF_CONFIG)

X_word_train = word_vec.fit_transform(X_clean_train)
X_word_test = word_vec.transform(X_clean_test)

scaler = StandardScaler(**SCALER_CONFIG)
X_eng_train_scaled = scaler.fit_transform(X_eng_train.values)
X_eng_test_scaled = scaler.transform(X_eng_test.values)

X_train = hstack([X_word_train, csr_matrix(X_eng_train_scaled)], format='csr')
X_test = hstack([X_word_test, csr_matrix(X_eng_test_scaled)], format='csr')

print(f'Train: {len(y_train):,}  |  Test: {len(y_test):,}')
print(f'Feature matrix (train): {X_train.shape}')
print(f'Feature matrix (test) : {X_test.shape}')


## 5. Grid Search — SGD Classifier

This notebook now uses the same shared environment as the Random Search and Optuna notebooks:
- 3-fold stratified CV
- word TF-IDF + engineered features
- same class-weight candidate list
- same `loss='modified_huber'` SGD family
- `max_iter=500` to shorten iteration time


In [ ]:
grid_sgd = GridSearchCV(
    SGDClassifier(**SGD_GRID_MODEL_CONFIG),
    SGD_GRID_PARAM_GRID,
    scoring='f1',
    cv=search_cv,
    n_jobs=-1,
    verbose=1,
)
grid_sgd.fit(X_train, y_train)

best_sgd = grid_sgd.best_estimator_
y_pred_sgd = best_sgd.predict(X_test)
y_prob_sgd = best_sgd.predict_proba(X_test)[:, 1]

print(f'Best params : {grid_sgd.best_params_}')
print(f'Best CV F1  : {grid_sgd.best_score_:.4f}')
print('\n=== SGD Report ===')
print(classification_report(y_test, y_pred_sgd, target_names=['Not Toxic', 'Toxic']))
print(f'ROC-AUC : {roc_auc_score(y_test, y_prob_sgd):.4f}')
print(f'F1      : {f1_score(y_test, y_pred_sgd):.4f}')


## 6. Grid Search — Logistic Regression

The LR search now compares `solver in ['liblinear', 'saga']` and `penalty in ['l1', 'l2']` with `max_iter=500`, matching the other tuning notebooks.


In [ ]:
grid_lr = GridSearchCV(
    LogisticRegression(**LR_GRID_MODEL_CONFIG),
    LR_GRID_PARAM_GRID,
    scoring='f1',
    cv=search_cv,
    n_jobs=-1,
    verbose=1,
)
grid_lr.fit(X_train, y_train)

best_lr = grid_lr.best_estimator_
y_pred_lr = best_lr.predict(X_test)
y_prob_lr = best_lr.predict_proba(X_test)[:, 1]

print(f'Best params : {grid_lr.best_params_}')
print(f'Best CV F1  : {grid_lr.best_score_:.4f}')
print('\n=== Logistic Regression Report ===')
print(classification_report(y_test, y_pred_lr, target_names=['Not Toxic', 'Toxic']))
print(f'ROC-AUC : {roc_auc_score(y_test, y_prob_lr):.4f}')
print(f'F1      : {f1_score(y_test, y_pred_lr):.4f}')


## 7. Compare all result

In [ ]:
results = {
    'Tuned SGD': {
        'F1': f1_score(y_test, y_pred_sgd),
        'Precision': precision_score(y_test, y_pred_sgd),
        'Recall': recall_score(y_test, y_pred_sgd),
        'ROC-AUC': roc_auc_score(y_test, y_prob_sgd),
        'Best CV F1': grid_sgd.best_score_,
    },
    'Tuned LR': {
        'F1': f1_score(y_test, y_pred_lr),
        'Precision': precision_score(y_test, y_pred_lr),
        'Recall': recall_score(y_test, y_pred_lr),
        'ROC-AUC': roc_auc_score(y_test, y_prob_lr),
        'Best CV F1': grid_lr.best_score_,
    },
}

result_df = pd.DataFrame(results).T.round(4)
print('=== Model Comparison ===')
print(result_df.to_string())

result_df['F1'].plot(kind='bar', color=['steelblue', 'tomato'], figsize=(8, 4))
plt.title('Grid Search F1 Score Comparison')
plt.ylabel('F1 Score')
plt.xticks(rotation=15)
plt.ylim(0.5, 0.85)
plt.tight_layout()
plt.show()


## 8. Save Best Model

In [ ]:
# Choose the best tuned model and save method-specific artifacts.
best_f1_sgd = f1_score(y_test, y_pred_sgd)
best_f1_lr = f1_score(y_test, y_pred_lr)

if best_f1_lr >= best_f1_sgd:
    final_model = best_lr
    print(f'Winner: Logistic Regression (F1={best_f1_lr:.4f})')
else:
    final_model = best_sgd
    print(f'Winner: SGD Classifier (F1={best_f1_sgd:.4f})')

with open(GRID_SEARCH_WORD_VECTORIZER_PATH, 'wb') as f:
    pickle.dump(word_vec, f)
with open(GRID_SEARCH_SCALER_PATH, 'wb') as f:
    pickle.dump(scaler, f)
with open(GRID_SEARCH_BEST_MODEL_PATH, 'wb') as f:
    pickle.dump(final_model, f)

print(f'Saved: {GRID_SEARCH_BEST_MODEL_PATH}')
print(f'Saved: {GRID_SEARCH_WORD_VECTORIZER_PATH} / {GRID_SEARCH_SCALER_PATH}')
